In [2]:
from google.colab import files

uploaded = files.upload()

Saving train_data.txt to train_data (1).txt
Saving test_data.txt to test_data.txt
Saving test_data_solution.txt to test_data_solution.txt


In [3]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import string
import warnings

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
# Load training data
train_data = pd.read_csv(
    "train_data.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "GENRE", "DESCRIPTION"]
)

# Load test data
test_data = pd.read_csv(
    "test_data.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "DESCRIPTION"]
)

# Load test solution
test_solution = pd.read_csv(
    "test_data_solution.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "GENRE", "DESCRIPTION"]
)

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [5]:
# Display first 5 rows
train_data.head()

,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


In [6]:
# Function to clean text

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Apply cleaning

train_data["DESCRIPTION"] = train_data["DESCRIPTION"].apply(clean_text)
test_data["DESCRIPTION"] = test_data["DESCRIPTION"].apply(clean_text)
test_solution["DESCRIPTION"] = test_solution["DESCRIPTION"].apply(clean_text)

print("Text cleaned successfully!")

Text cleaned successfully!


In [7]:
# Convert text into numerical features

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train = vectorizer.fit_transform(train_data["DESCRIPTION"])
X_test = vectorizer.transform(test_data["DESCRIPTION"])

y_train = train_data["GENRE"]
y_test = test_solution["GENRE"]

print("TF-IDF Vectorization Completed!")
print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

TF-IDF Vectorization Completed!
Training Shape: (54214, 5000)
Testing Shape: (54200, 5000)


In [8]:
# Train Logistic Regression model

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model Trained Successfully!")

Model Trained Successfully!


In [9]:
# Predict genres

predictions = model.predict(X_test)

print("Prediction Completed!")

Prediction Completed!


In [10]:
# Calculate Accuracy

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

Accuracy: 0.5830073800738007

Classification Report:

              precision    recall  f1-score   support

      action       0.48      0.29      0.36      1314
       adult       0.59      0.24      0.34       590
   adventure       0.59      0.16      0.26       775
   animation       0.49      0.06      0.11       498
   biography       0.00      0.00      0.00       264
      comedy       0.53      0.58      0.56      7446
       crime       0.36      0.04      0.07       505
 documentary       0.67      0.85      0.75     13096
       drama       0.54      0.77      0.64     13612
      family       0.49      0.09      0.15       783
     fantasy       0.56      0.06      0.10       322
   game-show       0.91      0.51      0.66       193
     history       0.00      0.00      0.00       243
      horror       0.64      0.57      0.60      2204
       music       0.67      0.45      0.54       731
     musical       0.27      0.02      0.04       276
     mystery       0.30    

In [11]:
# Display Confusion Matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

[[  379     2     8     1     0   128     7   140   467     1     0     0
      0    45     1     2     0     0     2     2    25    45    14     1
     33     0    11]
 [    6   141    20     0     0   179     0    47   142     0     0     0
      0     6     1     0     0     0     1     1     1    44     1     0
      0     0     0]
 [   32    39   127     7     0   111     0   132   209     6     5     0
      0    28     0     0     1     0     5     0    12    37     2     0
     11     0    11]
 [   27     0    14    31     0   125     0    75    94    17     6     1
      0    20     2     0     0     0     0     0    25    56     2     0
      3     0     0]
 [    0     0     0     0     0    18     0   172    62     0     0     0
      0     1     2     0     0     0     0     0     0     7     2     0
      0     0     0]
 [   57     8     7     4     0  4349     3   452  2117     9     0     2
      0    82    11     0     1     1    21     8    16   252     5     9
     14

In [12]:
# Test with your own movie description

sample = ["A young superhero fights powerful villains to save the world."]

sample_vector = vectorizer.transform(sample)

prediction = model.predict(sample_vector)

print("Predicted Genre:", prediction[0])

Predicted Genre: action


In [13]:
import joblib

joblib.dump(model, "movie_genre_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!
